In [ ]:
import os
import pandas as pd
import pywt
import wfdb
import numpy as np
from scipy.signal import butter,filtfilt,medfilt,resample,iirnotch

import matplotlib.pyplot as plt

SCD_PATH = os.environ.get("SCD_PATH", os.path.join("data", "SCD"))
NSR_PATH = os.environ.get("NSR_PATH", os.path.join("data", "NSR"))

# 根据表格创建时间配置字典
record_config = {
    '30': {'type': 'SCD',       'start': '6:54:33',     'end': '7:54:33'},
    '31': {'type': 'SCD',       'start': '12:42:24',    'end': '13:42:24'},
    '32': {'type': 'SCD',       'start': '15:45:18',    'end': '16:45:18'},
    '33': {'type': 'SCD',       'start': '3:46:19',     'end': '4:46:19'},
    '34': {'type': 'SCD',       'start': '5:35:44',     'end': '6:35:44'},
    '35': {'type': 'SCD',       'start': '23:34:56',    'end': '24:34:56'},
    '36': {'type': 'SCD',       'start': '17:59:01',    'end': '18:59:01'},
    '37': {'type': 'SCD',       'start': '0:31:13',     'end': '1:31:13'},
    '38': {'type': 'SCD',       'start': '7:01:54',     'end': '8:01:54'},
    '39': {'type': 'SCD',       'start': '3:37:51',     'end': '4:37:51'},
    '40': {'type': 'normal',   'start': 1*3600,        'end': 2*3600},  # 特殊处理1-2小时
    '41': {'type': 'SCD',       'start': '1:59:24',     'end': '2:59:24'},
    '42': {'type': 'normal',    'start': 1*3600,        'end': 2*3600},
    '43': {'type': 'SCD',       'start': '14:37:11',    'end': '15:37:11'},
    '44': {'type': 'SCD',       'start': '18:38:45',    'end': '19:38:45'},
    '45': {'type': 'SCD',       'start': '17:09:17',    'end': '18:09:17'},
    '46': {'type': 'SCD',       'start': '2:41:47',     'end': '3:41:47'},
    '47': {'type': 'SCD',       'start': '5:13:01',     'end': '6:13:01'},
    '48': {'type': 'SCD',       'start': '1:29:40',     'end': '2:29:40'},
    '49': {'type': 'normal',    'start': 1*3600,        'end': 2*3600},
    '50': {'type': 'SCD',       'start': '10:45:43',    'end': '11:45:43'},
    '51': {'type': 'SCD',       'start': '21:58:23',    'end': '22:58:23'},
    '52': {'type': 'SCD',       'start': '1:32:40',     'end': '2:32:40'}
}

fs = 250
target_fs = 128

# ---------- 0) 处理NaN&0 ----------
def fix_nan_and_zero(sig):
    sig = np.array(sig, dtype=float)
    sig[np.isnan(sig)] = np.nanmedian(sig)
    zero_idx = np.where(sig == 0)[0]
    if len(zero_idx)>0:
        sig[zero_idx] = np.median(sig)
    return sig


# ---------- 1) 高通滤波去基线漂移 (0.5Hz) ----------
def highpass_filter(sig, fs=128, cutoff=0.5, order=4):
    b, a = butter(order, cutoff/(fs/2), btype='high')
    return filtfilt(b, a, sig)


# ---------- 2) 陷波滤波去工频噪声 50Hz ----------
def notch_filter(sig, fs=128, f0=50, Q=30):
    b, a = iirnotch(f0/(fs/2), Q)
    return filtfilt(b, a, sig)


# ---------- 3) 小波去噪(Wavelet Denoising) ----------
def wavelet_denoise(sig, wavelet='db6', level=4):
    coeffs = pywt.wavedec(sig, wavelet, level=level)
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    uthresh = sigma * np.sqrt(2*np.log(len(sig)))
    coeffs[1:] = [pywt.threshold(c, uthresh, mode='soft') for c in coeffs[1:]]
    return pywt.waverec(coeffs, wavelet)


# ---------- 🌟 终版完整预处理函数 ----------
def preprocess_signal(sig, fs=128):
    def check(x,name):
        if np.isnan(x).any() or np.isinf(x).any():
            print(f"❌ NaN detected after {name}")
            print("signal stats:", np.nanmin(x), np.nanmax(x), np.nanmean(x))
        return x
    sig = fix_nan_and_zero(sig)            # 处理坏点
    check(sig,"fix_nan_and_zero")
    sig = highpass_filter(sig, fs)         # 去基线漂移
    check(sig,"highpass_filter")
    sig = notch_filter(sig, fs)            # 工频抑制(50Hz)
    check(sig,"notch_filter")
    sig = wavelet_denoise(sig)             # 小波平滑去肌电噪声
    check(sig,"wavelet_denoise")
    return sig

def extract_5min(sig):
    total = len(sig)
    each = total//7
    out=[]
    for i in range(7):
        start = i*each
        mid = start + each//2 - 150*128  #150秒=5分钟一半长度=150*128点
        mid = max(mid,0)
        out.append(sig[mid:mid+300*128]) #5分钟=300秒
    return out  #返回7段长度38400点

def cut_into_2s(sig):
    seg=2*128  #256个采样
    return np.array([sig[i*seg:(i+1)*seg] for i in range(150)]) #150段


#=========== 处理 SCD 记录 =============#
def load_SCD():
    X,Y=[],[]
    for rec in sorted(record_config.keys()):
        path=os.path.join(SCD_PATH,rec)
        if not os.path.exists(path+".dat"): continue
        sig,field = wfdb.rdsamp(path,channels=[0])
        sig = sig[:,0]

        # 取指定1h区间
        s = record_config[rec]['start']
        e = record_config[rec]['end']

        if isinstance(s,str):
            h,m,s2 = map(float,s.split(":"))
            start_sec=h*3600+m*60+s2
            h,m,s2 = map(float,e.split(":"))
            end_sec=h*3600+m*60+s2
        else:
            start_sec=s; end_sec=e

        start=int(start_sec*fs)
        end=int(end_sec*fs)

        seg=sig[start:end]
        seg = np.array(seg, dtype=float)
        if np.isnan(seg).any():
            seg = pd.Series(seg).interpolate(limit_direction="both").bfill().ffill().values
        seg=resample(seg,int(len(seg)*target_fs/fs))

        seg=preprocess_signal(seg)

        if record_config[rec]['type']=="normal":
            # normal作为类0 取中间5分钟→150样本
            mid=len(seg)//2-150*128
            mseg=seg[mid:mid+300*128]
            X.append(cut_into_2s(mseg))
            Y.append(np.zeros(150,dtype=int))
        else:
            # SCD →7类
            parts=extract_5min(seg)
            xp=[]; yp=[]
            for cls,p in enumerate(parts):
                xp.append(cut_into_2s(p))
                yp.append(np.ones(150)*cls)
            X.append(xp)          #(7,150,256)
            Y.append(yp)
    
    SCD_X=[];SCD_Y=[];NORM_X=[];NORM_Y=[]
    for i,key in enumerate(record_config.keys()):
        if i>=len(X):break
        if record_config[key]['type']=="normal":
            NORM_X.append(X[i])
            NORM_Y.append(Y[i])
        else:
            SCD_X.append(X[i])
            SCD_Y.append(Y[i])

    SCD_X=np.array(SCD_X) #(num,7,150,256)
    SCD_Y=np.array(SCD_Y)
    NORMAL_X=np.array(NORM_X)
    NORMAL_Y=np.array(NORM_Y)

    np.save("SCD_X.npy",SCD_X)
    np.save("SCD_Y.npy",SCD_Y)
    np.save("NORMAL_X.npy",NORMAL_X)
    np.save("NORMAL_Y.npy",NORMAL_Y)

    print("📌 SCD & normal 处理完成！")
    print("SCD_X:",SCD_X.shape,"SCD_Y:",SCD_Y.shape)
    print("NORMAL_X:",NORMAL_X.shape,"NORMAL_Y:",NORMAL_Y.shape)
    return


#=========== NSR处理 =============#
def load_NSR():
    X=[];Y=[]
    files=[f.replace(".dat","") for f in os.listdir(NSR_PATH) if f.endswith(".dat")]
    for rec in sorted(files):
        path=os.path.join(NSR_PATH,rec)
        sig,_=wfdb.rdsamp(path,channels=[0])
        sig=sig[:,0]
        sig=resample(sig,int(len(sig)*target_fs/fs))
        sig=preprocess_signal(sig)
        
        mid=len(sig)//2-150*128
        seg=sig[mid:mid+300*128]
        X.append(cut_into_2s(seg))
        Y.append(np.zeros(150))

    X=np.array(X);Y=np.array(Y)
    np.save("NSR_X.npy",X)
    np.save("NSR_Y.npy",Y)
    print("📌 NSR 处理完成！",X.shape,Y.shape)
    return





In [ ]:
#========== —— 运行入口 —— ==========#
load_SCD()
load_NSR()

In [ ]:
import numpy as np

# 1️⃣ 读取
SCD_X = np.load("SCD_X.npy")    # (20,7,150,256)
SCD_Y = np.load("SCD_Y.npy")    # (20,7,150)

NORMAL_X = np.load("NORMAL_X.npy")   # (3,150,256)
NORMAL_Y = np.load("NORMAL_Y.npy")   # (3,150)

NSR_X = np.load("NSR_X.npy")  # (18,150,256)
NSR_Y = np.load("NSR_Y.npy")  # (18,150)


# 2️⃣ SCD 展开为普通样本格式
SCD_X_flat = SCD_X.reshape(-1,150,256)   # (20*7=140 ,150,256)
SCD_Y_flat = SCD_Y.reshape(-1,150)       # (140 ,150)


# 3️⃣ 合并全部数据
X_total = np.vstack([SCD_X_flat, NORMAL_X, NSR_X])
Y_total = np.vstack([SCD_Y_flat, NORMAL_Y, NSR_Y])

print("合并完成")
print("X_total:", X_total.shape)  # → (?,150,256)
print("Y_total:", Y_total.shape)  # → (?,150)

# 4️⃣ 保存
np.save("X_total_150_256.npy", X_total)
np.save("Y_total_150.npy", Y_total)

print("\n已保存 → X_total_150_256.npy / Y_total_150.npy")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# 读取数据
X = np.load("X_total_150_256.npy")   # (161,150,256)
Y = np.load("Y_total_150.npy")       # (161,150)

print("数据加载成功:", X.shape, Y.shape)

# 随机抽一个样本
idx = random.randint(0, X.shape[0]-1)
signal = X[idx]   # shape (150,256)
label = Y[idx]    # shape (150,)

print(f" index = {idx}")

# -------- 可视化1：展示其中某一段 ECG 原波形 ----------
segment_id = 10  # 可修改 0-149
plt.figure(figsize=(10,4))
plt.plot(signal[segment_id])
plt.title(f"ECG Segment {segment_id}  (Label={label[segment_id]})")

plt.axis('off')   # 关闭坐标轴、刻度、边框、网格
plt.show()


# -------- 可视化2：拼接展示完整150段 ECG 波形 ----------
full_ecg = signal.reshape(-1) [:1000] # 150×256展平为38400点

plt.figure(figsize=(12,4))
plt.plot(full_ecg)
plt.title(f"ECG index={idx}")
plt.xlabel("Samples (150×256)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()



In [ ]:
import numpy as np

X_all_denoised = np.load("X_total_150_256.npy")
y_all = np.load("Y_total_150.npy")

print("加载完成")
print("X_all_denoised 形状:", X_all_denoised.shape)
print("y_all 形状:", y_all.shape)
print("y_all 分布:", {cls: np.sum(y_all == cls) for cls in np.unique(y_all)})

In [ ]:
import torch
X_all_denoised = torch.tensor(X_all_denoised, dtype=torch.float32)
print(torch.isnan(X_all_denoised).sum(), torch.isinf(X_all_denoised).sum())

nan_per_sample = torch.isnan(X_all_denoised).view(X_all_denoised.size(0), -1).sum(dim=1)
print("NaN per sample:", nan_per_sample[:])  # 先看前50条
print("坏样本数量:", (nan_per_sample > 0).sum().item())

total = X_all_denoised.size(0)
bad = (nan_per_sample > 0).sum().item()

print(f"总样本 {total} 条")
print(f"坏样本 {bad} 条，占比 {bad/total*100:.2f}%")



In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# 按 7:3 划分数据集（保持类别比例）
X_train, X_test, y_train, y_test = train_test_split(
    X_all_denoised,
    y_all,
    test_size=0.25,      # 75% 训练集，25% 测试集
    random_state=42,    # 固定随机种子，保证可复现
    stratify=y_all      # 保证类别比例不变
)

print("训练集样本数:", X_train.shape[0])
print("测试集样本数:", X_test.shape[0])

# 查看类别分布
unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)

print("\n训练集类别分布:")
for u, c in zip(unique_train, counts_train):
    print(f"类别 {u}: {c} 条样本")

print("\n测试集类别分布:")
for u, c in zip(unique_test, counts_test):
    print(f"类别 {u}: {c} 条样本")


In [ ]:
print("X_train 形状:", X_train.shape)
print("y_train 形状:", y_train.shape)

print("X_test 形状:", X_test.shape)
print("y_test 形状:", y_test.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter,filtfilt
import pywt
import random

#================ 参数 ==================
target_class = 6      # ⭐你想查看哪类 (0~6)
fs = 256              # 采样率
t = np.linspace(0,2,256)
#========================================


# 兼容 torch 或 numpy 数据
X = X_train.cpu().numpy() if hasattr(X_train,"cpu") else X_train
y = np.array(y_train)

# 找到该类全部样本并随机抽取
idx = np.argwhere(y==target_class)
if len(idx)==0:
    print("该类没有数据，请换 target_class")
    raise SystemExit

sample, seg = random.choice(idx)
sig = X[sample, seg, :]


#============ ECG滤波、去基线漂移 =============
def bandpass(sig,low=0.5,high=40,fs=256):
    b,a = butter(4,[low/(fs/2),high/(fs/2)],btype='band')
    return filtfilt(b,a,sig)

ecg = bandpass(sig,fs=fs)


#============ 小波CWT =============
scales = np.arange(1,80)    # 小波尺度，可调
coef, freqs = pywt.cwt(ecg, scales, 'morl', 1/fs)


#============ 画图 =============
plt.figure(figsize=(10,4))

# (1) ECG信号
plt.plot(t, ecg, color='black', linewidth=1.2)
plt.title(f"ECG")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(False)


plt.show()

print(f"已随机展示类别 {target_class} → sample:{sample}, seg:{seg}")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, stft
import pywt
import random

# STFT 参数（可根据论文需求微调）
f, tt, Zxx = stft(
    ecg,
    fs=fs,
    window='hann',
    nperseg=64,
    noverlap=48,
    nfft=256,
    boundary=None
)

plt.figure(figsize=(6, 4))
plt.pcolormesh(
    tt,
    f,
    np.abs(Zxx),
    shading='gouraud',
    cmap='jet'
)

plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.ylim(0, 40)   # ECG 有效频段
plt.colorbar(label="Magnitude")

plt.tight_layout()
plt.savefig("FFT.svg", format="svg", dpi=300)
plt.show()


print(f"已随机展示类别 {target_class} → sample:{sample}, seg:{seg}")
print("已额外保存 FFT 二维时频图：FFT.svg")

In [ ]:
import torch
X_train = torch.tensor(X_train, dtype=torch.float32)
print(torch.isnan(X_train).sum(), torch.isinf(X_train).sum())

### 2.1 时频转换（STFT + CWT）

In [ ]:
import torch
import numpy as np
from scipy.signal import stft
import pywt

def compute_stft(signal, fs=128, nperseg=64, noverlap=36):
    f, t, Zxx = stft(signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    return np.abs(Zxx)  # (freq_bins, time_steps)

def compute_cwt(signal, scales=None, wavelet='morl'):
    if scales is None:
        scales = np.linspace(1, 32, 32)  # 32 个尺度 → H=32
    coefficients, _ = pywt.cwt(signal, scales, wavelet)
    return np.abs(coefficients)  # (H, W)


In [ ]:
from tqdm import tqdm

def generate_tf_dataset(X_segmented, fs=128):
    tf_data = []
    for i in tqdm(range(X_segmented.shape[0]), desc="生成时频图"):
        segments = []
        for seg in X_segmented[i]:
            tf_map = compute_stft(seg, fs=fs)
            segments.append(tf_map)
        tf_data.append(segments)
    return np.array(tf_data)  # (样本数, 时间段数, freq_bins, time_steps)


def generate_wavelet_dataset(X_segmented, normalize=True, to_float32=True):
    wavelet_data = []
    for i in tqdm(range(X_segmented.shape[0]), desc="生成小波变换"):
        segments=[]
        for seg in X_segmented[i]:

            if isinstance(seg, torch.Tensor):
                seg = seg.detach().cpu().numpy()

            if normalize:
                seg = (seg - seg.mean()) / (seg.std()+1e-8)

            coeff = compute_cwt(seg)

            if to_float32:
                coeff = coeff.astype(np.float32)

            segments.append(coeff)

        wavelet_data.append(np.array(segments))

    return np.array(wavelet_data, dtype=object) # 若每段shape一致可改为float32



In [ ]:
X_train_time = X_train      # (120, 150, 256)
y_train_time = y_train      # (120*150,)
X_test_time = X_test        # (41, 150, 256)
y_test_time = y_test        # (41*150,)
print("训练集时间视角形状:", X_train_time.shape)
print("测试集时间视角形状:", X_test_time.shape) 


In [ ]:
X_train_tf = generate_tf_dataset(X_train_time)
X_test_tf = generate_tf_dataset(X_test_time)
print("train快速傅里叶变化形状:", X_train_tf.shape)
print("test快速傅里叶变化形状:", X_test_tf.shape)

In [ ]:
X_train_wavelet = generate_wavelet_dataset(X_train)
X_test_wavelet = generate_wavelet_dataset(X_test)
print("train小波变换形状:", X_train_wavelet.shape)
print("test小波变换形状:", X_test_wavelet.shape)

In [ ]:
import numpy as np
import torch

# 将 object 数组里的每个元素转换成 float32
X_train_wavelet_float = np.empty((120, 150, 32, 256), dtype=np.float32)
for i in range(120):
    for j in range(150):
        X_train_wavelet_float[i,j] = np.array(X_train_wavelet[i,j], dtype=np.float32)

# 转为 torch tensor
X_train_wavelet = torch.tensor(X_train_wavelet_float, dtype=torch.float32)

# 检查是否存在 NaN 或 Inf
print(torch.isnan(X_train_wavelet).sum(), torch.isinf(X_train_wavelet).sum())

In [ ]:
X_train_time = torch.tensor(X_train_time, dtype=torch.float32)
X_train_tf = torch.tensor(X_train_tf, dtype=torch.float32)

print(torch.isnan(X_train_time).sum(), torch.isinf(X_train_time).sum())
print(torch.isnan(X_train_tf).sum(), torch.isinf(X_train_tf).sum())
print(torch.isnan(X_train_wavelet).sum(), torch.isinf(X_train_wavelet).sum())


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

############# ① TCN处理 (B,150,256) → (B,150,256_out) #############
class TemporalTCN(nn.Module):
    def __init__(self, in_channels=256, num_channels=[128,128,256]):
        super().__init__()
        layers=[]
        for i,ch in enumerate(num_channels):
            layers.append(nn.Sequential(
                nn.Conv1d(in_channels if i==0 else num_channels[i-1],
                          ch, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.BatchNorm1d(ch)
            ))
        self.net = nn.Sequential(*layers)

    def forward(self,x):
        # x = (B,150,256) → 拼回CNN格式
        x = x.permute(0,2,1)        # → (B,256,150)
        x = self.net(x)             # → (B,256_out,150)
        return x.permute(0,2,1)     # → (B,150,256_out)


############# ② 时频 CNN (B,150,33,11) → (B,150,128) #############
class TimeFrequencyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

    def forward(self,x):
        B,W,H,Wd = x.shape              # (120,150,33,11)
        x = x.view(B*W,1,H,Wd)          # 合并窗口 → (120*150,1,33,11)
        x = self.cnn(x).flatten(1)      # → (B*W,128)
        return x.view(B,W,128)          # → (B,150,128)


############# ③ 小波 Transformer (B,150,32,256) → (B,150,128) #############
class WaveletTransformer(nn.Module):
    def __init__(self,embed=128,patch=16):
        super().__init__()
        self.patch=patch
        self.proj=nn.Linear(patch*patch,embed)
        encoder=nn.TransformerEncoderLayer(embed,4,batch_first=True)
        self.trans=nn.TransformerEncoder(encoder,2)
        self.cls=nn.Parameter(torch.zeros(1,1,embed))

    def forward(self,x):
        B,W,H,Wd=x.shape                 # (120,150,32,256)
        x=x.view(B*W,1,H,Wd)

        # 分patch
        x=x.unfold(2,self.patch,self.patch).unfold(3,self.patch,self.patch)
        x=x.contiguous().view(B*W,-1,self.patch*self.patch)  # (BW,N,P²)

        x=self.proj(x)                    # (BW,N,128)
        cls=self.cls.expand(B*W,1,-1)
        x=torch.cat([cls,x],1)
        x=self.trans(x)[:,0]              # cls token输出

        return x.view(B,W,128)            # → (B,150,128)


############# ④ 多模态融合输出 (B,150,512) #############
class MultiViewECGNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.tcn   = TemporalTCN()
        self.cnn   = TimeFrequencyCNN()
        self.wave  = WaveletTransformer()

        self.fuse_dim = 256+128+128
        # self.proj = nn.Linear(self.fuse_dim,128)   # 节点特征维度缩放

    def forward(self,x_time,x_tf,x_wave):
        f1 = self.tcn(x_time)               # (B,150,256)
        f2 = self.cnn(x_tf)                 # (B,150,128)
        f3 = self.wave(x_wave)              # (B,150,128)

        feat = torch.cat([f1,f2,f3],dim=-1) # (B,150,512)
        return feat                   # → (B,150,512) 作为 ODE-GCN输入


In [ ]:
import torch
device = torch.device(f"cuda:{1}" if torch.cuda.is_available() else "cpu")

X_train_time     = torch.tensor(X_train_time, dtype=torch.float32)
X_train_tf       = torch.tensor(X_train_tf, dtype=torch.float32)
X_train_wavelet  = torch.tensor(X_train_wavelet, dtype=torch.float32)


model = MultiViewECGNet().to(device)
X_train_time     = X_train_time.to(device)
X_train_tf       = X_train_tf.to(device)
X_train_wavelet  = X_train_wavelet.to(device)


model = MultiViewECGNet().to(device)

# 1. Wavelet 输入降采样（关键）
X_train_wavelet = F.interpolate(X_train_wavelet, size=(32,64))

# 2. 使用 mini-batch 前向
batch_size = 4

# 3. 逐batch提取特征
node_feats=[]
for i in range(0,len(X_train_time),batch_size):
    f=model(X_train_time[i:i+batch_size],X_train_tf[i:i+batch_size],X_train_wavelet[i:i+batch_size]).cpu()

    node_feats.append(f)

node_features=torch.cat(node_feats)
print(node_features.shape)


In [ ]:
print(torch.isnan(node_features).sum(), torch.isinf(node_features).sum())

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdiffeq import odeint_adjoint as odeint
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

# ===================== 构建邻接矩阵 =====================
def build_adj(x, k=10):
    B, N, C = x.size()
    x_norm = F.normalize(x, dim=-1)
    sim = torch.matmul(x_norm, x_norm.transpose(1, 2))
    topk = torch.topk(sim, k=k, dim=-1).indices
    adj = torch.zeros_like(sim)
    for b in range(B):
        adj[b].scatter_(1, topk[b], 1)
    deg = adj.sum(-1, keepdim=True) + 1e-8
    adj = adj / deg
    return adj

# ===================== GCN Layer =====================
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1, use_bn=True):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.use_bn = use_bn
        self.dropout = nn.Dropout(dropout)
        if use_bn:
            self.bn = nn.LayerNorm(out_dim)

    def forward(self, x, adj):
        h = torch.matmul(adj, x)
        h = self.fc(h)
        if self.use_bn:
            h = self.bn(h)
        h = F.relu(h)
        h = self.dropout(h)
        return h

# ===================== DeepGCN =====================
class DeepGCN(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128, num_layers=5, dropout=0.3):
        super().__init__()
        self.layers = nn.ModuleList()
        self.residual_projs = nn.ModuleList()
        dims = [input_dim] + [hidden_dim] * num_layers
        for i in range(num_layers):
            self.layers.append(GCNLayer(dims[i], dims[i+1], dropout=dropout))
            if dims[i] != dims[i+1]:
                self.residual_projs.append(nn.Linear(dims[i], dims[i+1]))
            else:
                self.residual_projs.append(nn.Identity())
        self.use_residual = True

    def forward(self, x, adj):
        h = x
        for layer, proj in zip(self.layers, self.residual_projs):
            h_in = proj(h)     # 残差投影
            h = layer(h, adj)  # GCN 计算
            h = h + h_in       # 残差连接
        return h

# ===================== ODEFunc =====================
class ODEFunc(nn.Module):
    def __init__(self, gcn_model, adj):
        super().__init__()
        self.gcn = gcn_model
        self.register_buffer('adj', adj)

    def forward(self, t, x):
        dx = self.gcn(x, self.adj)
        return dx

# ===================== ODEBlock =====================
class ODEBlock(nn.Module):
    def __init__(self, odefunc, t=torch.tensor([0,1.0])):
        super().__init__()
        self.odefunc = odefunc
        self.register_buffer('integration_time', t)  # 注册 buffer，保持 device

    def forward(self, x):
        t = self.integration_time.to(x.device)  # 保证 t 和 x 同设备
        out = odeint(self.odefunc, x, t, method='rk4')
        return out[-1]


# ===================== ODE-GCN =====================
class ODEGCN(nn.Module):
    def __init__(self, input_dim=128, hidden_dim=128, num_layers=3, num_classes=7, dropout=0.2):
        super().__init__()
        self.gcn = DeepGCN(input_dim, hidden_dim, num_layers, dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, adj):
        ode_func = ODEFunc(self.gcn, adj)
        ode_block = ODEBlock(ode_func)
        h = ode_block(x)
        out = self.classifier(h)
        return out  # [B,N,num_classes]


In [ ]:
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score

import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "cpu"


y_train_time  = torch.tensor(y_train_time, dtype=torch.float32)
node_features = node_features.clone().detach().float().to(device)  # 确保无梯度、float
y_train_time  = y_train_time.clone().detach().long().to(device)    # ⚠绝对不要 float

# 可选：构建 DataLoader，支持 batch 训练
dataset = TensorDataset(node_features, y_train_time)
loader = DataLoader(dataset, batch_size=16, shuffle=True)  # 根据显存调batch_size

# ===================== 训练循环 =====================
model = ODEGCN(
    input_dim=512,  # 512
    hidden_dim=512,
    num_layers=7,
    num_classes=7,
    dropout=0.3
).to(device)

# ===================== 优化器 & 损失 & 调度器 =====================
initial_lr = 1e-4  # 调小学习率
optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr)
loss_fn = nn.CrossEntropyLoss()

# Reduce LR on Plateau
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=20, verbose=True
)

max_grad_norm = 1.0  # 梯度裁剪阈值

best_f1 = 0.0
# 额外保存一个 near-best 模型
saved_mid_f1_model = False
mid_f1_low = 0.985
mid_f1_high = 0.99
mid_model_path = "mid_f1_odegcn_model.pt"
best_model_path = "best_odegcn_model.pt"


num_epochs = 500

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    for batch_features, batch_labels in loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        with torch.no_grad():
            adj = build_adj(batch_features, k=10).to(device)

        pred = model(batch_features, adj)
        loss = loss_fn(pred.view(-1, 7), batch_labels.view(-1))

        optimizer.zero_grad()
        loss.backward()

        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        total_loss += loss.item() * batch_features.size(0)
        pred_labels = pred.argmax(dim=-1)
        total_correct += (pred_labels == batch_labels).sum().item()
        total_samples += batch_labels.numel()

        all_preds.append(pred_labels.cpu())
        all_labels.append(batch_labels.cpu())

    avg_loss = total_loss / len(dataset)
    acc = total_correct / total_samples
    all_preds = torch.cat(all_preds).numpy().flatten()
    all_labels = torch.cat(all_labels).numpy().flatten()
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall    = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro')

    print(
    f"Epoch {epoch+1}/{num_epochs} | "
    f"Loss: {avg_loss:.4f} | "
    f"Acc: {acc:.4f} | "
    f"Precision: {precision:.4f} | "
    f"Recall: {recall:.4f} | "
    f"F1: {f1:.4f}"
)


    # 保存 F1 最好的模型
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_model_path)
        print(f"🏆 Best model saved at epoch {epoch+1} with F1={f1:.4f}")
    
    if (mid_f1_low <= f1 < mid_f1_high) and (not saved_mid_f1_model):
        torch.save(model.state_dict(), mid_model_path)
        saved_mid_f1_model = True
        print(
            f"📦 Mid-F1 model saved at epoch {epoch+1} "
            f"with F1={f1:.4f} (range {mid_f1_low}-{mid_f1_high})"
        )

    # 调整学习率
    scheduler.step(avg_loss)



In [ ]:
# ===================== 绘制混淆矩阵 =====================
best_model = ODEGCN(
    input_dim=512,
    hidden_dim=512,
    num_layers=7,
    num_classes=7,
    dropout=0.3
).to(device)
best_model.load_state_dict(torch.load(best_model_path))
best_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch_features, batch_labels in loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        adj = build_adj(batch_features, k=10).to(device)
        pred = best_model(batch_features, adj)
        pred_labels = pred.argmax(dim=-1)
        all_preds.append(pred_labels.cpu())
        all_labels.append(batch_labels.cpu())

all_preds = torch.cat(all_preds).numpy().flatten()
all_labels = torch.cat(all_labels).numpy().flatten()

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig("confusion_matrix.svg", format="svg", dpi=300)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ================= 数据 =================
precision = [98.28, 98.62, 98.91, 99.27, 98.00,
             98.79, 99.17, 98.91, 98.45, 98.95]

accuracy = [98.43, 98.66, 99.02, 99.33, 97.86,
            98.84, 99.21, 99.02, 98.52, 98.94]

recall = [98.39, 98.53, 99.13, 99.27, 97.91,
          98.87, 99.11, 99.13, 98.67, 98.80]

f1 = [98.33, 98.57, 99.02, 99.27, 97.95,
      98.83, 99.14, 99.02, 98.56, 98.87]

data = [precision, accuracy, recall, f1]
labels = ['Precision', 'Accuracy', 'Recall', 'F1-score']

# ================= 统计量 =================
means = [np.mean(d) for d in data]
stds  = [np.std(d, ddof=1) for d in data]

# ================= 画箱线图 =================
plt.figure(figsize=(7, 5))

box = plt.boxplot(
    data,
    labels=labels,
    patch_artist=True,
    showmeans=True,
    meanline=True
)

# 颜色（论文友好）
colors = ['lightblue', 'lightgreen', 'lightcoral', 'khaki']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

for median in box['medians']:
    median.set(color='black', linewidth=1.5)

for mean_line in box['means']:
    mean_line.set(color='red', linewidth=1.5)

all_values = np.concatenate(data)

mean_y = all_values.max() + 0.1   # Mean 统一高度
std_y  = all_values.min() - 0.1   # Std 统一高度

# ================= 标注 Mean & Std =================
for i, (mean, std, values) in enumerate(zip(means, stds, data), start=1):
    q3 = np.percentile(values, 75)
    q1 = np.percentile(values, 25)

    # Mean —— 箱体上方
    plt.text(
        i,
        mean_y-0.25,
        f" { mean:.2f}%",
        ha='center',
        va='bottom',
        fontsize=11,
        color='black'
    )

    # Std —— 箱体下方（红色）
    plt.text(
        i,
        std_y+0.6,
        f" ± {std:.2f}%",
        ha='center',
        va='top',
        fontsize=11,
        color='red'
    )

plt.ylabel('Performance (%)')
plt.title('Distribution of Performance Metrics over 10 Experiments')
plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("metrics_boxplot_mean_std_separated.svg", format="svg", dpi=300)
plt.show()



In [ ]:
class MultiViewECGNet(nn.Module):
    def __init__(self, num_classes=7):
        super(MultiViewECGNet, self).__init__()
        self.num_classes = num_classes
        self.tcn = TemporalTCN(in_channels=1, num_channels=[64, 128, 256])
        self.tf_cnn = TimeFrequencyCNN(in_channels=1)
        self.wavelet_trans = WaveletTransformer(embed_dim=128)
        self.feat_dim = 256 + 128 + 128
        # 修改输出层：输出证据向量（使用Softplus确保非负）
        self.fc_evidence = nn.Linear(self.feat_dim, num_classes)
        
        # 移除原来的不确定性输出层
        # self.fc_log_var = nn.Linear(self.feat_dim, 1)

    def forward(self, x_time, x_tf, x_wavelet):
        f1 = self.tcn(x_time)         # (B, 256)
        f2 = self.tf_cnn(x_tf)        # (B, 128)
        f3 = self.wavelet_trans(x_wavelet)  # (B, 128)

        feat = torch.cat([f1, f2, f3], dim=1)
        # 输出证据向量，使用Softplus确保非负
        evidence = F.softplus(self.fc_evidence(feat))  # (B, num_classes)
        return evidence

In [ ]:
y_train_time.max()

In [ ]:
assert y_train_time.min() >= 0
assert y_train_time.max() < 7
print("标签合法 ✅")

In [ ]:
y_train_time.shape

In [ ]:
X_train_time.shape

In [ ]:
from torch.utils.data import Dataset, DataLoader
class MultiViewDataset(Dataset):
    def __init__(self, X_time, X_tf, X_wavelet, y):
        self.X_time = torch.tensor(X_time, dtype=torch.float32)
        self.X_tf = torch.tensor(X_tf, dtype=torch.float32)
        self.X_wavelet = torch.tensor(X_wavelet, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.X_time.shape[0] * self.X_time.shape[1]

    def __getitem__(self, idx):
        sample_idx = idx // self.X_time.shape[1]
        segment_idx = idx % self.X_time.shape[1]

        x_time = self.X_time[sample_idx, segment_idx, :]                # (L,)
        x_tf = self.X_tf[sample_idx, segment_idx, :, :]                 # (freq_bins, time_steps)
        x_wavelet = self.X_wavelet[sample_idx, segment_idx, :, :]       # (coeff_length, L)
        y = self.y[sample_idx * self.X_time.shape[1] + segment_idx]

        x_time = x_time.unsqueeze(0)        # (1, L)
        x_tf = x_tf.unsqueeze(0)            # (1, freq_bins, time_steps)
        x_wavelet = x_wavelet.unsqueeze(0)  # (1, coeff_length, L)


        return x_time, x_tf, x_wavelet, y


train_dataset = MultiViewDataset(X_train_time, X_train_tf, X_train_wavelet, y_train_time)
test_dataset = MultiViewDataset(X_test_time, X_test_tf, X_test_wavelet, y_test_time)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
# 修复的KL散度计算
def kl_divergence_loss(alpha, targets, num_classes=7):
    """
    稳定版本的KL散度计算
    alpha: 狄利克雷参数, (B, C)
    targets: one-hot标签, (B, C)
    """
    # 构建调整后的alpha_tilde
    # 对于真实类别: alpha_tilde = 1
    # 对于其他类别: alpha_tilde = alpha
    alpha_tilde = targets + (1 - targets) * alpha
    
    # 目标分布: 均匀狄利克雷分布 Dir(1,1,...,1)
    alpha_prior = torch.ones_like(alpha)
    
    # 计算总浓度参数
    S_tilde = torch.sum(alpha_tilde, dim=1, keepdim=True)
    S_prior = torch.sum(alpha_prior, dim=1, keepdim=True)
    
    # 使用稳定的对数伽马函数计算
    # KL(Dir(alpha_tilde) || Dir(alpha_prior))
    # = log Γ(S_tilde) - log Γ(S_prior) 
    #   - Σ[log Γ(alpha_tilde_i) - log Γ(alpha_prior_i)]
    #   + Σ[(alpha_tilde_i - alpha_prior_i) * (ψ(alpha_tilde_i) - ψ(S_tilde))]
    
    # 计算各项
    term1 = torch.lgamma(S_tilde) - torch.lgamma(S_prior)
    term2 = torch.sum(torch.lgamma(alpha_prior) - torch.lgamma(alpha_tilde), dim=1, keepdim=True)
    term3 = torch.sum((alpha_tilde - alpha_prior) * 
                     (torch.digamma(alpha_tilde) - torch.digamma(S_tilde)), dim=1, keepdim=True)
    
    kl_div = term1 + term2 + term3
    return torch.mean(kl_div)

In [ ]:
# 修复的EDL损失函数
def edl_mse_loss(evidence, targets, num_classes=7, annealing_coeff=0.01):
    """
    修复的EDL MSE损失函数
    """
    # 计算狄利克雷分布参数
    alpha = evidence + 1.0  # (B, C)
    
    # 总浓度参数
    S = torch.sum(alpha, dim=1, keepdim=True)  # (B, 1)
    
    # 预测概率
    P = alpha / S  # (B, C)
    
    # MSE损失项 - 保持稳定
    mse_term = torch.sum((targets - P) ** 2, dim=1)  # (B,)
    
    # 方差正则化项
    var_term = torch.sum(P * (1 - P) / (S + 1), dim=1)  # (B,)
    
    mse_loss = torch.mean(mse_term + var_term)
    
    # KL正则化项 - 使用修复的版本
    kl_loss = kl_divergence_loss(alpha, targets, num_classes)
    
    # 使用较小的权重，避免KL损失主导
    total_loss = mse_loss + annealing_coeff * kl_loss
    
    return total_loss, mse_loss, kl_loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, recall_score, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
import scipy.special as special

device = torch.device(f"cuda:{1}" if torch.cuda.is_available() else "cpu")

def evaluate_multiview_model(model, data_loader, device=device):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0

    all_preds = []
    all_labels = []
    all_uncertainties = []
    all_probabilities = []

    with torch.no_grad():
        for x_time, x_tf, x_wavelet, y_batch in data_loader:
            x_time = x_time.to(device)
            x_tf = x_tf.to(device)
            x_wavelet = x_wavelet.to(device)
            y_batch = y_batch.to(device)
            
            # 将标签转换为one-hot编码
            targets_one_hot = F.one_hot(y_batch, num_classes=7).float()

            # 前向传播，得到证据向量
            evidence = model(x_time, x_tf, x_wavelet)  # (B, 7)
            
            # 计算狄利克雷参数
            alpha = evidence + 1.0  # (B, 7)
            S = torch.sum(alpha, dim=1, keepdim=True)  # (B, 1)
            
            # 计算预测概率和不确定性
            probabilities = alpha / S  # (B, 7)
            uncertainty = 7.0 / S.squeeze(1)  # (B,)
            
            # 使用预测概率计算损失
            loss, mse_loss, kl_loss = edl_mse_loss(evidence, targets_one_hot, annealing_coeff=0.01)

            loss_sum += loss.item() * y_batch.size(0)
            _, predicted = torch.max(probabilities.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            all_uncertainties.extend(uncertainty.detach().cpu().numpy())
            all_probabilities.extend(probabilities.detach().cpu().numpy())

    acc = correct / total
    avg_loss = loss_sum / total

    # 计算指标
    f1 = f1_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    uncertainties = np.array(all_uncertainties)

    # 混淆矩阵
    cm = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()

    print(f"Accuracy: {acc:.4f} | F1-score: {f1:.4f} | Recall: {recall:.4f} | "
          f"Avg Loss: {avg_loss:.4f} | Avg Uncertainty: {uncertainties.mean():.4f} | "
          f"min: {uncertainties.min():.4f} max: {uncertainties.max():.4f}")

    return acc, avg_loss, f1, recall, all_labels, all_preds, uncertainties, all_probabilities



def train_multiview_model(model, train_loader, test_loader, y_train, num_epochs=20, lr=1e-3, device=device, class_names=None):
    model = model.to(device)

    # 计算类别权重
    class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)  # 添加权重衰减

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    mse_losses, kl_losses = [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_mse_loss = 0.0
        running_kl_loss = 0.0
        correct = 0
        total = 0

        all_preds = []
        all_labels = []
        all_uncertainties = []

        # 使用固定的较小的退火系数，避免KL损失主导
        annealing_coeff = 0.01  # 固定小值

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for x_time, x_tf, x_wavelet, y_batch in loop:
            x_time = x_time.to(device)
            x_tf = x_tf.to(device)
            x_wavelet = x_wavelet.to(device)
            y_batch = y_batch.to(device)
            
            # 将标签转换为one-hot编码
            targets_one_hot = F.one_hot(y_batch, num_classes=7).float()

            optimizer.zero_grad()
            
            # 前向传播，得到证据向量
            evidence = model(x_time, x_tf, x_wavelet)  # (B, 7)
            
            # 计算EDL损失
            loss, mse_loss, kl_loss = edl_mse_loss(evidence, targets_one_hot, annealing_coeff=annealing_coeff)
            
            # 梯度裁剪，避免梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * y_batch.size(0)
            running_mse_loss += mse_loss.item() * y_batch.size(0)
            running_kl_loss += kl_loss.item() * y_batch.size(0)
            
            # 计算预测结果
            alpha = evidence + 1.0
            S = torch.sum(alpha, dim=1, keepdim=True)
            probabilities = alpha / S
            uncertainty = 7.0 / S.squeeze(1)
            
            _, predicted = torch.max(probabilities.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

            all_preds.append(predicted.cpu())
            all_labels.append(y_batch.cpu())
            all_uncertainties.extend(uncertainty.detach().cpu().numpy())

            loop.set_postfix(
                loss=running_loss/total, 
                mse_loss=running_mse_loss/total,
                kl_loss=running_kl_loss/total,
                acc=correct/total
            )

        train_loss = running_loss / total
        train_mse_loss = running_mse_loss / total
        train_kl_loss = running_kl_loss / total
        train_acc = correct / total
        
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        mse_losses.append(train_mse_loss)
        kl_losses.append(train_kl_loss)

        # 拼接并转为 numpy
        all_labels = torch.cat(all_labels).numpy()
        all_preds = torch.cat(all_preds).numpy()

        f1 = f1_score(all_labels, all_preds, average="macro")
        recall = recall_score(all_labels, all_preds, average="macro")

        uncertainties = np.array(all_uncertainties)

        if (epoch + 1) % 20 == 0:
            # 混淆矩阵
            cm = confusion_matrix(all_labels, all_preds)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm)
            disp.plot(cmap=plt.cm.Blues)
            plt.title("Confusion Matrix")
            plt.show()

        print(f"\nEpoch {epoch+1} — Train Loss: {train_loss:.4f} "
              f"(MSE: {train_mse_loss:.4f}, KL: {train_kl_loss:.4f}), "
              f"Train Acc: {train_acc:.4f}, "
              f"Uncertainty mean: {uncertainties.mean():.4f}, std: {uncertainties.std():.4f}, "
              f"min: {uncertainties.min():.4f}, max: {uncertainties.max():.4f}, "
              f"F1: {f1:.4f}, Recall: {recall:.4f}")

        # 验证集评估（可选）
        # if test_loader is not None and epoch % 20 == 0:
            # val_acc, val_loss, val_f1, val_recall, val_labels, val_preds, val_uncertainties, val_probs = evaluate_multiview_model(model, test_loader, device)
            # val_accs.append(val_acc)
            # val_losses.append(val_loss)

    return model, train_losses, train_accs, mse_losses, kl_losses, uncertainties, all_labels, all_preds



In [ ]:
# 创建并训练模型
model = MultiViewECGNet(num_classes=7)
model.to(device)

model, train_losses, train_accs, mse_losses, kl_losses, uncertainties, all_labels, all_preds = train_multiview_model(
    model, train_loader, test_loader, y_train_time, num_epochs=2000, lr=1e-3, device=device
)